## If you use Google Colab

1. Open this notebook in Colab.
2. In Colab, click `File -> Save a copy in Drive` so you can edit your own copy.
3. Run the setup cell below to clone the repository and install the dependencies.


In [ ]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/MarcEricP/IntroDL.git"
REPO_DIR = Path("/content/image-segmentation")
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    if not REPO_DIR.exists():
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")

    os.chdir(REPO_DIR)
    print(f"Working directory: {Path.cwd()}")
    get_ipython().system("pip install -q -r requirements.txt")
else:
    print("Local environment detected: no clone or pip install needed.")


## Exercise version\n\nThis notebook is a student version of the tutorial. Several key pieces of code have been replaced by prompts such as `# complete here...`. Fill them in, then rerun the cell before moving on.\n

## Image segmentation

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.morphology import binary_dilation, disk
from skimage.segmentation import find_boundaries

frangipani_image_path = Path("dataset") / "train" / "image" / "frangipani_00885.jpg"
frangipani_mask_path = Path("dataset") / "train" / "mask" / "frangipani_00885.tif"

frangipani_image = imread(frangipani_image_path)
frangipani_mask = imread(frangipani_mask_path)
frangipani_outline = find_boundaries(frangipani_mask, mode="outer")
frangipani_outline = binary_dilation(frangipani_outline, disk(2))

overlay = frangipani_image.copy()
overlay[frangipani_outline] = [255, 0, 0]

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(overlay)
ax.set_title("Image segmentation")
ax.axis("off")
plt.show()


## Oxford Flowers dataset

We first need a dataset before we can talk about segmentation. The Oxford Flowers dataset gives us many examples of flowers together with their labels, and here we work with a smaller segmentation subset built from it.

Starting with a mosaic is useful because it immediately answers two questions:
- what kinds of objects are we trying to segment?
- how much visual variability is there across species?


In [ ]:
dataset_image_dir = Path("dataset") / "train" / "image"
species_examples = {}

# We keep one representative image per species so that the mosaic stays compact and readable.
for image_file in sorted(dataset_image_dir.glob("*.jpg")):
    species_name = " ".join(image_file.stem.split("_")[:-1])
    if species_name not in species_examples:
        species_examples[species_name] = image_file

n_species = len(species_examples)
ncols = 4
nrows = int(np.ceil(n_species / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = np.array(axes).reshape(-1)

for ax, (species_name, image_file) in zip(axes, species_examples.items()):
    ax.imshow(imread(image_file))
    ax.set_title(species_name.title(), fontsize=12)
    ax.axis("off")

for ax in axes[len(species_examples):]:
    ax.axis("off")

fig.suptitle("Flower species in our subset of the Oxford Flowers dataset", fontsize=16)
plt.tight_layout()
plt.show()


## Pixels in an array

Before segmenting an image, it is worth going back to the most basic representation: an image is just an array of numbers. Each position in that array corresponds to one pixel.

Looking at one pixel value is a simple way to connect the visual object we see on the screen with the numerical data that an algorithm actually receives.


In [ ]:
import numpy as np

image_path = Path("dataset") / "train" / "image" / "passion_flower_00005.jpg"
mask_path = Path("dataset") / "train" / "mask" / "passion_flower_00005.tif"
image = imread(image_path)
ground_truth_mask = imread(mask_path)

# Choose a pixel location and inspect its RGB value.
# x, y = ...
# pixel_value = ...
raise NotImplementedError("Complete the pixel access lines above, then rerun the cell.")


## Channels

A color image does not store one number per pixel, but one number per channel at each pixel. In an RGB image, the channels are red, green, and blue.

We inspect the shape because array layout matters in image analysis. Then we rearrange the axes into the `(channel, height, width)` convention, which is common in machine learning libraries such as PyTorch.


In [ ]:
print(f"Original shape (height, width, channel): {image.shape}")

# Rearrange the image to the channel-first convention used later in the notebook.
# image_chw = ...
raise NotImplementedError("Complete the channel rearrangement above.")


The image is composed of three channels. Displaying them separately helps us see that each channel carries related but not identical information.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
channel_names = ["Red channel", "Green channel", "Blue channel"]

for channel_index, ax in enumerate(axes):
    ax.imshow(image_chw[channel_index], cmap="gray")
    ax.set_title(channel_names[channel_index])
    ax.axis("off")

plt.show()


In microscopy and many scientific imaging settings, we often work with a single informative channel. To keep the tutorial simple, we now retain only the red channel and treat it as a grayscale image.


In [ ]:
# Keep only one channel and display it as a grayscale image.
# gray = ...
raise NotImplementedError("Complete the single-channel extraction above.")


## Goal: automatically get the mask of the flower

This dataset already provides a binary ground-truth mask for the same image. We use that mask for two reasons:
- it tells us what the correct answer is
- it lets us measure whether our segmentation methods are doing a good job


In [ ]:
# We highlight one foreground pixel and one background pixel to make the binary task explicit.
foreground_pixel = np.argwhere(ground_truth_mask)[1000]
background_pixel = np.argwhere(~ground_truth_mask)[100000]
fx, fy = foreground_pixel
bx, by = background_pixel

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title("Grayscale input")
axes[1].imshow(ground_truth_mask, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Ground-truth binary mask")
axes[1].annotate(
    "This pixel should be classified as object",
    xy=(fy, fx),
    xytext=(fy + 70, fx - 60),
    color="red",
    fontsize=11,
    arrowprops=dict(arrowstyle="->", color="red", lw=2),
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="red", alpha=0.9),
)
axes[1].annotate(
    "This pixel should be classified as background",
    xy=(by, bx),
    xytext=(by + 70, bx + 50),
    color="red",
    fontsize=11,
    arrowprops=dict(arrowstyle="->", color="red", lw=2),
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="red", alpha=0.9),
)
for ax in axes:
    ax.axis("off")
plt.show()


## A first try: simple threshold on the intensity

A threshold is the simplest segmentation rule: pixels darker or brighter than a given value are assigned to one class. This is useful pedagogically because it shows how a full segmentation map can already be produced from a very simple pixel-wise rule.

Otsu's method automatically chooses a threshold from the image histogram.


In [ ]:
from skimage.filters import threshold_otsu

# Compute an automatic threshold and turn it into a binary mask.
# threshold = ...
# mask = ...
raise NotImplementedError("Complete the thresholding code above.")


## Performance metric: intersection over union

Once we have a predicted mask, we need a way to quantify how good it is. Intersection over union (IoU), also called the Jaccard index, measures how much the predicted object region overlaps with the true one.

This is a natural metric for segmentation because it directly compares two regions, not just individual pixels.


In [ ]:
# IoU compares the overlap area with the union area.
def intersection_over_union(pred_mask, true_mask):
    # complete here...
    raise NotImplementedError("Complete the IoU computation.")

# This overlay makes errors easy to read visually.
def make_mask_overlay(pred_mask, true_mask):
    overlay = np.zeros(pred_mask.shape + (3,), dtype=float)
    overlay[..., 0] = pred_mask.astype(float)
    overlay[..., 1] = true_mask.astype(float)
    overlay[..., 2] = pred_mask.astype(float)
    return overlay

test_image_paths = sorted((Path("dataset") / "test" / "image").glob("passion_flower_*.jpg"))
test_mask_paths = [Path("dataset") / "test" / "mask" / f"{image_path.stem}.tif" for image_path in test_image_paths]

threshold_results = []
for image_path, mask_path in zip(test_image_paths, test_mask_paths):
    image_rgb = imread(image_path)
    image_gray = image_rgb[..., 0]
    true_mask = imread(mask_path).astype(bool)
    # complete here...
    raise NotImplementedError("Complete the threshold baseline inside the loop.")


## Pixel classifier

Thresholding uses only one number per pixel: the intensity. A classical pixel classifier is more flexible: for each pixel, we compute a richer feature vector and let a machine-learning model decide whether the pixel belongs to the object or the background.

This is close to the spirit of tools such as ilastik: first build informative handcrafted features, then train a supervised classifier.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from skimage.feature import multiscale_basic_features

train_image_paths = sorted((Path("dataset") / "train" / "image").glob("passion_flower_*.jpg"))
train_mask_paths = [Path("dataset") / "train" / "mask" / f"{image_path.stem}.tif" for image_path in train_image_paths]
test_image_paths = sorted((Path("dataset") / "test" / "image").glob("passion_flower_*.jpg"))
test_mask_paths = [Path("dataset") / "test" / "mask" / f"{image_path.stem}.tif" for image_path in test_image_paths]

train_image = imread(train_image_paths[0])
train_gray = train_image[..., 0]
train_mask = imread(train_mask_paths[0]).astype(bool)

# Compute multiscale handcrafted features for one training image.
# train_features = ...
raise NotImplementedError("Complete the multiscale feature computation above.")


In [ ]:
# Now we train the pixel classifier on all training images.
train_feature_blocks = []
train_label_blocks = []
for image_path, mask_path in zip(train_image_paths, train_mask_paths):
    image_rgb = imread(image_path)
    image_gray = image_rgb[..., 0]
    mask_bool = imread(mask_path).astype(bool)
    # feature_block = ...
    raise NotImplementedError("Complete the feature extraction inside the loop.")

# X_train = ...
# y_train = ...
# sample training pixels and fit the random forest here.
raise NotImplementedError("Complete the random-forest training block.")


In [ ]:
# We now test the classifier on unseen images and compare prediction against ground truth.
pixel_classifier_results = []
for image_path, mask_path in zip(test_image_paths, test_mask_paths):
    image_rgb = imread(image_path)
    image_gray = image_rgb[..., 0]
    true_mask = imread(mask_path).astype(bool)
    feature_map = multiscale_basic_features(
        image_gray,
        intensity=True,
        edges=True,
        texture=True,
        sigma_min=1,
        sigma_max=8,
        channel_axis=None,
    )
    pred_mask = classifier.predict(feature_map.reshape(-1, feature_map.shape[-1])).reshape(image_gray.shape)
    iou = intersection_over_union(pred_mask, true_mask)
    pixel_classifier_results.append((image_path.stem, image_gray, pred_mask, true_mask, iou))

global_pixel_classifier_iou = np.mean([result[-1] for result in pixel_classifier_results])
fig, axes = plt.subplots(len(pixel_classifier_results), 2, figsize=(8, 4 * len(pixel_classifier_results)))
axes = np.atleast_2d(axes)

for row, (name, image_gray, pred_mask, true_mask, iou) in enumerate(pixel_classifier_results):
    axes[row, 0].imshow(image_gray, cmap="gray")
    axes[row, 0].set_title(name.replace("_", " ").title())
    axes[row, 1].imshow(make_mask_overlay(pred_mask, true_mask))
    axes[row, 1].set_title(f"Prediction (magenta) vs ground truth (green)\nIoU = {iou:.3f}")

for ax in axes.ravel():
    ax.axis("off")

fig.suptitle(f"Global IoU: {global_pixel_classifier_iou:.3f}", fontsize=16)
plt.tight_layout()
plt.show()


## Deep learning: a UNet for flower segmentation

We now switch from handcrafted features to a neural network. In the classical pipeline above, we chose the features ourselves. In deep learning, the model learns useful internal representations directly from the training images.

The plan is the same as in many practical deep-learning projects: convert images to tensors, define a model, build a dataloader, choose a loss and an optimizer, train the model, and finally evaluate what it predicts.

To keep the example simple, we use only the red channel and learn a binary segmentation task: foreground flower versus background.


In [ ]:
import sys
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

if str(Path(".").resolve()) not in sys.path:
    sys.path.append(str(Path(".").resolve()))

from unet import UNet
from unet_resnet import UNetWithResnet50Encoder

torch.manual_seed(0)

sample_image_path = sorted((Path("dataset") / "train" / "image").glob("*.jpg"))[0]
sample_image_rgb = imread(sample_image_path)
sample_red = sample_image_rgb[..., 0].astype(np.float32) / 255.0

# Convert the NumPy array to a PyTorch tensor and add the channel dimension.
# sample_red_tensor = ...
raise NotImplementedError("Complete the tensor conversion above.")


### Construct the dataloader

Neural networks are trained in batches, and all tensors in the same batch must have the same spatial size. That is why we resize images and masks before stacking them together.

For the test set, we also keep the original image and mask. This is useful for teaching: later, we will show the original image and original ground truth, and only resize the network prediction back to that resolution for visualization and IoU.


In [ ]:
IMAGE_SIZE = 256
TRAIN_BATCH_SIZE = 4
TEST_BATCH_SIZE = 1

# Images are resized with bilinear interpolation because they are continuous-valued.
def resize_image_tensor(image_tensor, image_size=IMAGE_SIZE):
    # complete here...
    raise NotImplementedError("Complete image resizing.")

# Masks are resized with nearest-neighbor interpolation so that they stay binary.
def resize_mask_tensor(mask_tensor, image_size=IMAGE_SIZE):
    # complete here...
    raise NotImplementedError("Complete mask resizing.")

class FlowerSegmentationDataset(Dataset):
    def __init__(self, split, image_size=IMAGE_SIZE, keep_original=False):
        self.image_paths = sorted((Path("dataset") / split / "image").glob("*.jpg"))
        self.mask_paths = [Path("dataset") / split / "mask" / f"{image_path.stem}.tif" for image_path in self.image_paths]
        self.image_size = image_size
        self.keep_original = keep_original

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_rgb = imread(self.image_paths[index])
        mask_np = imread(self.mask_paths[index]).astype(np.float32)

        original_image = torch.from_numpy(image_rgb[..., 0].astype(np.float32) / 255.0).unsqueeze(0)
        original_mask = torch.from_numpy(mask_np).unsqueeze(0)
        # resized_image = ...
        # resized_mask = ...
        raise NotImplementedError("Complete the resized image/mask creation.")


### Model definition: a single convolutional layer

Before using a full UNet, it is useful to look at one convolutional layer in isolation. A convolution applies the same small trainable filter everywhere in the image. This is what makes convolutional networks good at detecting repeated local patterns such as edges, blobs, and textures.


In [ ]:
# Four filters produce four different feature maps from the same input image.
conv_layer = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=5, padding=2)

with torch.no_grad():
    # conv_outputs = ...
    raise NotImplementedError("Apply the convolutional layer to the input tensor.")


### Model definition: UNet

A UNet is an encoder-decoder architecture. The encoder gradually compresses the image into higher-level features, while the decoder progressively rebuilds a segmentation map. Skip connections pass spatial detail from the encoder to the decoder, which is especially useful for segmentation.

We can use two variants:
- the light UNet from [`unet.py`](/home/menir/ml_course/image-segmentation/unet.py)
- the heavier ResNet-backbone UNet from [`unet_resnet.py`](/home/menir/ml_course/image-segmentation/unet_resnet.py)

For a tutorial, the light version is the best default because it is faster to train and easier to inspect.


In [ ]:
# By default we keep the lighter model so that the tutorial runs faster.
USE_RESNET_BACKBONE = False

def build_unet(use_resnet_backbone=USE_RESNET_BACKBONE):
    # complete here...
    raise NotImplementedError("Return the correct UNet architecture here.")

MODEL_NAME = "resnet_unet" if USE_RESNET_BACKBONE else "light_unet"
untrained_unet = build_unet(USE_RESNET_BACKBONE)
print(f"Model selected: {MODEL_NAME}")
print(untrained_unet)
print(f"Number of parameters: {sum(parameter.numel() for parameter in untrained_unet.parameters()):,}")

# The model returns raw logits; we apply the sigmoid only when probabilities are needed.
def forward_logits(model, images):
    # complete here...
    raise NotImplementedError("Return the logits from the network.")


In [ ]:
# We choose a few representative layers for visualization.
def activation_layers_for_model(model):
    if hasattr(model, "input_block"):
        return [
            ("Input block", lambda model: model.input_block),
            ("Encoder block 1", lambda model: model.down_blocks[0]),
            ("Encoder block 2", lambda model: model.down_blocks[1]),
            ("Decoder block -2", lambda model: model.up_blocks[-2]),
            ("Decoder block -1", lambda model: model.up_blocks[-1]),
        ]
    return [
        ("Down block 1", lambda model: model.dconv_down1),
        ("Down block 2", lambda model: model.dconv_down2),
        ("Down block 3", lambda model: model.dconv_down3),
        ("Up block 2", lambda model: model.dconv_up2),
        ("Up block 1", lambda model: model.dconv_up1),
    ]

activation_sample = test_dataset[0]
activation_image = activation_sample["image"]

def capture_activation_maps(model, image_tensor, device):
    activations = {}
    hooks = []

    for layer_name, layer_getter in activation_layers_for_model(model):
        module = layer_getter(model)

        def hook(module, inputs, output, layer_name=layer_name):
            activations[layer_name] = output.detach().cpu()

        hooks.append(module.register_forward_hook(hook))

    model = model.to(device)
    model.eval()
    with torch.no_grad():
        _ = forward_logits(model, image_tensor.unsqueeze(0).to(device))

    for hook in hooks:
        hook.remove()

    return activations

def plot_activation_maps(model, image_tensor, device, title):
    activations = capture_activation_maps(model, image_tensor, device)
    fig, axes = plt.subplots(len(activations), 4, figsize=(12, 3 * len(activations)))
    axes = np.atleast_2d(axes)

    for row, (layer_name, activation_tensor) in enumerate(activations.items()):
        activation_tensor = activation_tensor[0]
        channel_indices = np.linspace(0, activation_tensor.shape[0] - 1, 4, dtype=int)

        for col, channel_index in enumerate(channel_indices):
            axes[row, col].imshow(activation_tensor[channel_index].numpy(), cmap="magma")
            axes[row, col].set_title(f"{layer_name}\nchannel {channel_index}")
            axes[row, col].axis("off")

    fig.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


### Optimizer: Adam and the training loop

The loss tells us how far the prediction is from the target. The optimizer uses that information to update the network weights. Adam is a popular default because it adapts the update size for each parameter automatically.

In each training iteration we follow the same pattern:
1. clear old gradients with `optimizer.zero_grad()`
2. predict a batch
3. compute the Dice loss
4. backpropagate with `loss.backward()`
5. update the weights with `optimizer.step()`


In [ ]:
LEARNING_RATE = 1e-3
CPU_EPOCHS = 1
GPU_EPOCHS = 100
CHECKPOINT_EVERY = 10
CHECKPOINT_DIR = Path("checkpoints")
SAVED_CHECKPOINT_PATH = CHECKPOINT_DIR / "saved_unet_latest.pt"
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Dice loss directly measures overlap, which is often more informative than plain BCE for sparse masks.
def dice_loss(logits, targets, smooth=1.0):
    # complete here...
    raise NotImplementedError("Complete the Dice loss implementation.")

def save_checkpoint(model, optimizer, epoch, loss_history, checkpoint_path):
    torch.save(
        {
            "epoch": epoch,
            "model_name": MODEL_NAME,
            "use_resnet_backbone": USE_RESNET_BACKBONE,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss_history": loss_history,
        },
        checkpoint_path,
    )

def latest_checkpoint_path():
    latest_path = CHECKPOINT_DIR / "unet_latest.pt"
    if latest_path.exists():
        return latest_path
    epoch_paths = sorted(CHECKPOINT_DIR.glob("unet_epoch_*.pt"))
    return epoch_paths[-1] if epoch_paths else None

def load_trained_unet(checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = build_unet(checkpoint.get("use_resnet_backbone", False))
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(device)
    model.eval()
    return model, checkpoint

def train_model(model, loader, optimizer, device, num_epochs):
    model = model.to(device)
    loss_history = []

    if device.type == "cuda":
        torch.cuda.synchronize(device)
    start_time = time.perf_counter()

    for epoch in range(num_epochs):
        model.train()
        progress = tqdm(loader, desc=f"Epoch {epoch + 1}/{num_epochs}", leave=False)
        for batch in progress:
            images = batch["image"].to(device)
            masks = batch["mask"].to(device)

            # Clear previous gradients, run the model, compute the loss, backpropagate, update.
            # 1. set gradients to zero
            

            # 2. compute output
            

            # 3. compute loss
            

            # 4. compute gradients (backpropagation)
            

            # 5. update parameters
            

            raise NotImplementedError("Complete the five training steps above.")

        if (epoch + 1) % CHECKPOINT_EVERY == 0:
            epoch_checkpoint_path = CHECKPOINT_DIR / f"unet_epoch_{epoch + 1:03d}.pt"
            save_checkpoint(model, optimizer, epoch + 1, loss_history, epoch_checkpoint_path)

    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elapsed_seconds = time.perf_counter() - start_time

    final_checkpoint_path = CHECKPOINT_DIR / "unet_latest.pt"
    save_checkpoint(model, optimizer, num_epochs, loss_history, final_checkpoint_path)
    return model, loss_history, elapsed_seconds, final_checkpoint_path


### Launch the training on the CPU

We first run the training on the CPU to make the computational cost visible. This is useful pedagogically: even a small UNet is already much heavier than the classical pixel classifier.


In [ ]:
cpu_device = torch.device("cpu")
cpu_unet = build_unet(USE_RESNET_BACKBONE)
cpu_optimizer = torch.optim.Adam(cpu_unet.parameters(), lr=LEARNING_RATE)

# Let's train the network on the CPU.
# cpu_unet, cpu_loss_history, cpu_elapsed_seconds, cpu_checkpoint_path = train_model(
#     # complete here...
# )
raise NotImplementedError("Complete the CPU training call above.")


### Switch everything to the GPU and relaunch the training

The training code itself barely changes. We just move the model and the batches to `cuda`, and the same loop usually runs much faster.

In Google Colab, you can request a GPU with `Runtime -> Change runtime type -> Hardware accelerator -> GPU`. Then reconnect to the runtime and rerun the notebook from the top.

Access to a GPU in Colab is not guaranteed: it depends on availability and on your account type.


In [ ]:
if torch.cuda.is_available():
    gpu_device = torch.device("cuda")
    gpu_unet = build_unet(USE_RESNET_BACKBONE)
    gpu_optimizer = torch.optim.Adam(gpu_unet.parameters(), lr=LEARNING_RATE)

    # Relaunch the same training loop on the GPU.
    # gpu_unet, gpu_loss_history, gpu_elapsed_seconds, gpu_checkpoint_path = train_model(
    #     # complete here...
    # )
    raise NotImplementedError("Complete the GPU training call above.")
else:
    trained_checkpoint_path = SAVED_CHECKPOINT_PATH if SAVED_CHECKPOINT_PATH.exists() else None
    trained_device = cpu_device

    if trained_checkpoint_path is not None:
        trained_unet, loaded_checkpoint = load_trained_unet(trained_checkpoint_path, trained_device)
        training_loss_history = loaded_checkpoint.get("loss_history", cpu_loss_history)
        print(f"CUDA is not available, so the protected saved model {trained_checkpoint_path.name} was loaded on the CPU.")
    else:
        trained_unet = cpu_unet
        training_loss_history = cpu_loss_history
        print("CUDA is not available and checkpoints/saved_unet_latest.pt was not found, so the CPU model will be used for the next steps.")


### Download the checkpoints from Colab

Training can take time, so it is useful to keep a copy of the learned weights. On Colab, the following cell downloads the latest saved checkpoint directly from the runtime.


In [ ]:
if IS_COLAB:
    from google.colab import files

    checkpoint_path = latest_checkpoint_path()
    if checkpoint_path is None:
        print("No checkpoint was found yet.")
    else:
        files.download(str(checkpoint_path))
else:
    print("This download helper is only available in Google Colab.")


In [ ]:
# The loss curve helps us see whether optimization is progressing over time.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cpu_loss_history, label=f"CPU ({CPU_EPOCHS} epoch)")
if "gpu_loss_history" in globals():
    ax.plot(gpu_loss_history, label=f"GPU ({GPU_EPOCHS} epochs)")
ax.set_xlabel("Training step")
ax.set_ylabel("Dice loss")
ax.set_title("UNet training loss")
ax.legend()
plt.tight_layout()
plt.show()


### Prediction on the test set

Once the model is trained, we switch from learning to inference: we freeze the weights, run the network on unseen images, threshold the probabilities, and compare the predicted mask with the ground truth.

The UNet is trained on resized inputs. For visualization, we keep the original image and the original ground-truth mask, then resize the network prediction back to the original image size before plotting and computing the IoU.


### About resizing

The network receives resized images because all tensors in a batch must have the same spatial size. This makes batching and training simple.

For interpretation, however, we do not want to show only the resized view. That is why we keep the original image and original ground truth, and then resize the prediction back to the original resolution before displaying it.


In [ ]:
# During evaluation we keep the original image for display, but resize the prediction back before measuring overlap.
def collect_predictions(model, loader, device):
    model.eval()
    results = []
    image_iterator = tqdm(loader, desc="Predicting test images")
    with torch.no_grad():
        for batch in image_iterator:
            # complete here...
            raise NotImplementedError("Complete the prediction-and-resize block inside this loop.")
    return results


### What is under the hood? Activation maps at various depths

A trained network is not just a black box: each layer produces intermediate feature maps. Looking at a few channels at different depths helps us see how the representation changes from low-level patterns to more abstract structures.


In [ ]:
# We now visualize internal feature maps after training.
plot_activation_maps(trained_unet, activation_image, trained_device, "Activation maps at various depths after training")


## Explore and try to improve performance !

- change neural network backbone
- use three channels (rgb) instead of one
- train for longer 
- tweak learning rate
- ...